In [0]:
# ============================================================
# GOLD LAYER - DIM_ACCOUNT - CORRECTED VERSION
# ============================================================

from pyspark.sql import functions as F
from datetime import datetime

# ============================================================
# GET ENVIRONMENT FROM ADF
# ============================================================

try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from ADF: {env}")
except:
    env = "DEV"
    print(f"📌 Using default: {env}")

# ============================================================
# STORAGE ACCOUNT
# ============================================================

storage_account_name = "capstonestorage01"

# ============================================================
# CONTAINER NAMES BASED ON ENVIRONMENT
# ============================================================

if env == 'DEV':
    silver_container = 'silver-dev'
    gold_container = 'gold-dev'
elif env == 'TEST':
    silver_container = 'silver-test'
    gold_container = 'gold-test'
else:  # PROD
    silver_container = 'silver'
    gold_container = 'gold'

# ============================================================
# BUILD PATHS
# ============================================================

SILVER = f"abfss://{silver_container}@{storage_account_name}.dfs.core.windows.net"
GOLD = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 SILVER Container: {silver_container}")
print(f"📁 GOLD Container: {gold_container}")
print(f"📍 SILVER Path: {SILVER}")
print(f"📍 GOLD Path: {GOLD}")
print(f"{'='*55}\n")

# ============================================================
# LOAD FROM SILVER
# ============================================================

df_account = spark.read.format("delta").load(f"{SILVER}/account_master")
print(f"📖 Loaded account records: {df_account.count():,}")

# ============================================================
# TRANSFORM - CREATE DIM_ACCOUNT
# ============================================================

dim_account = df_account.select(
    F.col("account_id"),
    F.col("customer_id"),
    F.col("product_type"),
    F.col("account_open_date"),
    F.col("credit_limit"),
    F.col("interest_rate"),
    F.col("account_status"),
    
    # Credit tier based on credit limit
    F.when(F.col("credit_limit") < 200000, F.lit("Basic"))
     .when(F.col("credit_limit") < 500000, F.lit("Standard"))
     .when(F.col("credit_limit") < 800000, F.lit("Premium"))
     .otherwise(F.lit("Elite"))
     .alias("credit_tier"),
    
    # Rate band based on interest rate
    F.when(F.col("interest_rate") < 12, F.lit("Low Rate"))
     .when(F.col("interest_rate") < 18, F.lit("Mid Rate"))
     .otherwise(F.lit("High Rate"))
     .alias("rate_band"),
    
    F.current_timestamp().alias("gold_created_at")
)

print(f"🔄 Transformed rows: {dim_account.count():,}")

# ============================================================
# WRITE TO GOLD
# ============================================================

dim_account.write.format("delta").mode("overwrite").option("overwriteSchema", "true")\
    .save(f"{GOLD}/dim_account")

# ============================================================
# SUMMARY
# ============================================================

print(f"\n{'='*55}")
print(f"✅ dim_account : {dim_account.count():,} rows")
print(f"📁 Written to: {GOLD}/dim_account")
print(f"🏭 Environment: {env}")
print(f"{'='*55}")